# Bronze Layer — Raw Ingestion

Ingest vehicle registration data from data.gov.my and save as a Delta table.

- **Source:** Parquet file from storage.dosm.gov.my
- **Output:** `bronze_vehicle_registrations` Delta table
- **Rule:** No transformations — data is saved exactly as received, with only an `ingested_at` timestamp added.


In [1]:
from pyspark.sql import functions as F

BASE_URL = "https://storage.data.gov.my/transportation"
YEARS = [2025, 2026]
TABLE_NAME = "bronze_vehicle_registrations"

## Read raw data from source


In [2]:
# Read the full Parquet file directly from data.gov.my storage
# Using pandas as a bridge; Spark can't read from HTTPS URLs directly
import pandas as pd

dfs = []
for year in YEARS:
    url = f"{BASE_URL}/cars_{year}.parquet"
    try:
        pdf = pd.read_parquet(url)
        dfs.append(pdf)
        print(f"Loaded {year}: {len(pdf)} rows")
    except Exception as e:
        print(f"Skipped {year}: {e}")

pdf_all = pd.concat(dfs, ignore_index=True)
df_raw = spark.createDataFrame(pdf_all)

print(f"\nTotal rows loaded: {df_raw.count()}")
df_raw.printSchema()

Loaded 2025: 870327 rows
Loaded 2026: 67995 rows

Total rows loaded: 938322
root
 |-- date_reg: date (nullable = true)
 |-- type: string (nullable = true)
 |-- maker: string (nullable = true)
 |-- model: string (nullable = true)
 |-- colour: string (nullable = true)
 |-- fuel: string (nullable = true)
 |-- state: string (nullable = true)



## Add ingestion metadata and save to Delta


In [3]:
# Add ingested_at timestamp
df_bronze = df_raw.withColumn("ingested_at", F.current_timestamp())

# Write to Delta table
df_bronze.write.format("delta").mode("overwrite").saveAsTable(TABLE_NAME)

print(f"Written to table: {TABLE_NAME}")

Written to table: bronze_vehicle_registrations


## Verify the table


In [5]:
# Read back from the Delta table to confirm it worked
df_check = spark.table(TABLE_NAME)

print(f"Row count: {df_check.count()}")
print(f"Columns: {df_check.columns}")
df_check.show(5)

Row count: 938322
Columns: ['date_reg', 'type', 'maker', 'model', 'colour', 'fuel', 'state', 'ingested_at']
+----------+-------+------+-------+------+------+-----------+--------------------+
|  date_reg|   type| maker|  model|colour|  fuel|      state|         ingested_at|
+----------+-------+------+-------+------+------+-----------+--------------------+
|2025-07-31|motokar|Proton|Persona|silver|petrol|Rakan Niaga|2026-02-15 14:18:...|
|2025-07-31|motokar|Proton|Persona|  grey|petrol|Rakan Niaga|2026-02-15 14:18:...|
|2025-07-31|motokar|Proton|Persona|  grey|petrol|Rakan Niaga|2026-02-15 14:18:...|
|2025-07-31|motokar|Proton|Persona|  grey|petrol|Rakan Niaga|2026-02-15 14:18:...|
|2025-07-31|motokar|Proton|Persona|  grey|petrol|Rakan Niaga|2026-02-15 14:18:...|
+----------+-------+------+-------+------+------+-----------+--------------------+
only showing top 5 rows
